### Data Collection

#### Source of dataset
The dataset is given by RateBeer platform. We personally asked for a permission to give us for this project
#### Location of dataset
The dataset is in a zip format located in the same directory: beer_reviews_ratebeer.zip. 
#### Nature of data
The dataset is stored in a plain text file (when you extract .zip file). It has almost 2,9 million records. 
It has a format like this in the following

++++

beer/name: John Harvards Simcoe IPA

beer/beerId: 63836

beer/brewerId: 8481

beer/ABV: 5.4

beer/style: India Pale Ale &#40;IPA&#41;

review/appearance: 4/5

review/aroma: 6/10

review/palate: 4/5

review/taste: 7/10

review/overall: 13/20

review/time: 1157241600

review/profileName: TomDecapolis

review/text: On tap at the John Harvards in Springfield PA.  Pours a ruby red amber with a medium off whie creamy head that left light lacing.  Aroma of orange and various other citrus.  A little light for what I was expecting from this beers aroma...expecting more from the Simcoe.  Flavor of pine, orange, grapefruit and some malt balance.  Very light bitterness for the 80+ IBUs they said this one had.

++++

#### Extracting .zip dataset
This is the first step in this notebook. We are going to extract the text file from zip file. 
Please note that at the end, we are going to remove the extracted file to not leave any unwanted file in the system.

In [13]:
#Import libraries
import zipfile # to extract zip file
import os # to be able to store/remove extracted file

zip_file_path = 'beer_reviews_ratebeer.zip'
extracted_dir = 'dataset'
extracted_filename = 'beer_reviews_ratebeer.txt'

print(f'Extracting dataset from {zip_file_path} into {extracted_dir} with filename {extracted_filename}')
with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    # Extract all the contents to the specified directory
    zip_ref.extractall(extracted_dir)

extracted_file_path = os.path.join(extracted_dir, extracted_filename)
print(f'Completed, the file can be found in {extracted_file_path}')

Extracting dataset from beer_reviews_ratebeer.zip into dataset with filename beer_reviews_ratebeer.txt
Completed, the file can be found in dataset\beer_reviews_ratebeer.txt


#### Parsing dataset (txt) file into dataframe
This step is to parse all the fields in the text format and store in a dataframe. 
2.9 million records can take time, that's why we added some sort of percentage so that you would have an idea when it is going to be finished.
After this process, the dataframe is created with parsed data and saved as csv file.

In [14]:
!pip install -q pandas
from html import unescape #some beer style has html formatting
import pandas as pd

# Set the batch size based on your system's capabilities
batch_size = 10000
current_count = 0

total_lines = sum(1 for line in open(extracted_file_path, 'r', encoding='utf-8', errors='ignore'))
print(f'There are {total_lines} lines, and this process may take a long time.')

# Read and parse the data
with open(extracted_file_path, 'r', encoding='utf-8', errors='ignore') as file:
    current_review = {}
    batch_data = []
    for idx, line in enumerate(file, 1):
        # Skip empty lines
        if not line.strip():
            continue

        key, value = line.strip().split(':', 1)
        current_review[key] = value.strip()

        # Check if a complete record has been read
        if key == 'review/text':
            # Accept only beerId as int
            try:
                beerId = int(current_review.get('beer/beerId'))
            except ValueError:
                beerId = 0
            
            batch_data.append((
                current_review.get('beer/name'),
                beerId,
                int(current_review.get('beer/brewerId')),
                current_review.get('beer/ABV'),
                unescape(current_review.get('beer/style')),
                current_review.get('review/appearance'),
                current_review.get('review/aroma'),
                current_review.get('review/palate'),
                current_review.get('review/taste'),
                current_review.get('review/overall'),
                int(current_review.get('review/time')),
                current_review.get('review/profileName'),
                current_review.get('review/text')
            ))
            current_count = current_count + 1

            # Print the progress
            if current_count == batch_size:
                # Clear current count
                current_count = 0
                # Calculate the progress percentage
                progress_percentage = (idx / total_lines) * 100
                print(f'Progress: {progress_percentage:.1f}%', end='\r', flush=True)
# Print 100% when the loop is done
print('Completed, creating a dataframe!')

#Define columns for dataframe in the same order in the text file
columns = [
    'beer_name',
    'beer_id',
    'brewer_id',
    'beer_abv',
    'beer_style',
    'review_appearance',
    'review_aroma',
    'review_palate',
    'review_taste',
    'review_overall',
    'review_time',
    'review_profilename',
    'review_text'
]
df = pd.DataFrame(batch_data, columns=columns)

There are 40938282 lines, and this process may take a long time.
Completed, creating a dataframe!


In [15]:
#Final step to remove all leftovers
csv_file_path = os.path.join(extracted_dir, 'ratebeer_dataset_raw.csv')
print(f'Save dataframe as csv file {csv_file_path} for the next steps')
df.to_csv(csv_file_path, index=False)
print('Dataframe is saved in csv file!')
os.remove(extracted_file_path)
print(f'Remove raw text file {extracted_file_path}')

Save dataframe as csv file dataset\ratebeer_dataset_raw.csv for the next steps
Dataframe is saved in csv file!
Remove raw text file dataset\beer_reviews_ratebeer.txt


#### End of Step 1
In this step, we collected the data from a datasource, parsed it and properly saved in csv format
So it is ready for the next step.